# Model 4 - Gingival Inflammation (gingivitis) detection

Detects inflamed-gum regions (erythema / edema / visible bleeding) on intraoral & mobile photos.
Merges two verified CC BY 4.0 datasets into one `gingivitis` class, all views combined.

**Sources (both verified to annotate GUM tissue):**
- sampling/gingivitis-6uyts - 468 imgs, detection, 1 class
- Pranta/gum-disease-mmzcr - segmentation; Periodontitis dropped, only gingivitis kept

**Kaggle:** GPU T4, Internet On, Persistence = Variables and Files.
Run: Cell 1 -> RESTART KERNEL -> Cell 2 -> ... -> Cell 7.

**Result:** val mAP50 ~0.80. Lessons baked in below (no numpy pin, single GPU, guarded bundle).

## Cell 1 - install (NO numpy pin; use Kaggle's stock numpy 2.x)

In [ ]:
!pip -q install -U ultralytics roboflow
print("installed - now Run -> Restart Kernel, then run from Cell 2")

## Cell 2 - verify (RESTART KERNEL FIRST)

In [ ]:
import numpy, torch, ultralytics
print("numpy", numpy.__version__, "| ultralytics", ultralytics.__version__, "| GPUs", torch.cuda.device_count())

## Cell 3 - download both datasets (public Universe, no fork, no credits)

In [ ]:
from roboflow import Roboflow
API_KEY = "PASTE_YOUR_ROBOFLOW_KEY"   # app.roboflow.com/settings/api  (or use a Kaggle Secret)
rf = Roboflow(api_key=API_KEY)

SOURCES = {
    "sampling": ("keshav-raja-s-workspace", "gingivitis-6uyts-qbisk", 1),
    "pranta":   ("keshav-raja-s-workspace", "gum-disease-mmzcr", 1),
}
paths = {}
for key,(ws,proj,ver) in SOURCES.items():
    ds = rf.workspace(ws).project(proj).version(ver).download("yolov8")
    paths[key] = ds.location
    print(key, "->", ds.location)

## Cell 4 - merge -> one `gingivitis` class, keep only gingivitis boxes, save 6 check images

In [ ]:
import os, glob, shutil, yaml, random, cv2
from pathlib import Path

MERGED = "/kaggle/working/merged"
for sp in ("train","valid","test"):
    os.makedirs(f"{MERGED}/{sp}/images", exist_ok=True)
    os.makedirs(f"{MERGED}/{sp}/labels", exist_ok=True)
os.makedirs(f"{MERGED}/_check", exist_ok=True)

def split_dir(root, sp):
    for cand in (sp, "val" if sp=="valid" else sp):
        if (Path(root)/cand/"images").exists(): return Path(root)/cand
    return None

def read_names(root):
    y = yaml.safe_load(open(Path(root)/"data.yaml"))
    return y["names"] if isinstance(y["names"], list) else [y["names"][i] for i in sorted(y["names"])]

def to_bbox(coords):
    if len(coords) == 4: return coords
    xs = coords[0::2]; ys = coords[1::2]
    x0,x1 = min(xs),max(xs); y0,y1 = min(ys),max(ys)
    return [(x0+x1)/2, (y0+y1)/2, x1-x0, y1-y0]

counts={"train":0,"valid":0,"test":0}; boxes=0; kept=0
for src, root in paths.items():
    names=[n.lower() for n in read_names(root)]
    gid=[i for i,n in enumerate(names) if "ging" in n]
    print(f"{src}: classes={names} -> gingivitis ids={gid}")
    for sp in ("train","valid","test"):
        sd=split_dir(root,sp)
        if sd is None: continue
        for img in glob.glob(str(sd/"images"/"*")):
            stem=Path(img).stem; lbl=sd/"labels"/f"{stem}.txt"; out=[]
            if lbl.exists():
                for ln in open(lbl):
                    p=ln.split()
                    if len(p)<5: continue
                    if int(float(p[0])) not in gid: continue
                    xc,yc,w,h=to_bbox(list(map(float,p[1:])))
                    out.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"); boxes+=1
            if not out: continue
            ns=f"{src}_{stem}"
            shutil.copy(img,f"{MERGED}/{sp}/images/{ns}{Path(img).suffix}")
            open(f"{MERGED}/{sp}/labels/{ns}.txt","w").write("\n".join(out))
            counts[sp]+=1; kept+=1

open(f"{MERGED}/data.yaml","w").write(
    f"path: {MERGED}\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 1\nnames: ['gingivitis']\n")
print("\nMERGED:",counts,"| imgs:",kept,"| boxes:",boxes)

samp=random.sample(glob.glob(f"{MERGED}/train/images/*"), min(6,counts["train"]))
for i,ip in enumerate(samp):
    im=cv2.imread(ip); H,W=im.shape[:2]
    for ln in open(f"{MERGED}/train/labels/{Path(ip).stem}.txt"):
        _,xc,yc,w,h=map(float,ln.split())
        cv2.rectangle(im,(int((xc-w/2)*W),int((yc-h/2)*H)),(int((xc+w/2)*W),int((yc+h/2)*H)),(0,0,255),3)
    cv2.imwrite(f"{MERGED}/_check/sample_{i}.jpg",im)
print("WROTE 6 samples -> merged/_check/  - OPEN THEM: red boxes must be on GUMS, not teeth")

**STOP: open the 6 images in `merged/_check/`. Boxes on gums -> continue. Boxes on teeth -> reconsider the dataset.**

## Cell 5 - train (single-GPU fix: hide GPU 1 BEFORE torch init; absolute save path)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["RANK"]="-1"; os.environ["LOCAL_RANK"]="-1"
import torch
print("visible GPUs now:", torch.cuda.device_count())   # MUST be 1

from ultralytics import YOLO
PROJECT="/kaggle/working/m4"
model=YOLO("yolo11s.pt")
model.train(data=f"{MERGED}/data.yaml", epochs=120, imgsz=640, batch=16, device=0,
            patience=30, seed=42, project=PROJECT, name="gingivitis", exist_ok=True,
            close_mosaic=10, save_period=20, verbose=True)
RUN=f"{PROJECT}/gingivitis"
print("TRAIN_DONE | weights:", os.listdir(f"{RUN}/weights"))

## Cell 6 - test-split metrics + ONNX

In [ ]:
from ultralytics import YOLO
BEST=f"{RUN}/weights/best.pt"
m=YOLO(BEST)
mt=m.val(data=f"{MERGED}/data.yaml", split="test", imgsz=640, device=0)
print("TEST mAP50:",round(float(mt.box.map50),4),"| mAP50-95:",round(float(mt.box.map),4),
      "| P:",round(float(mt.box.mp),3),"| R:",round(float(mt.box.mr),3))
try: m.export(format="onnx",imgsz=640,opset=12); print("onnx ok")
except Exception as e: print("onnx skipped:",e)

## Cell 7 - bundle (guarded: refuses to zip if best.pt is missing)

In [ ]:
import shutil, os, glob
assert os.path.exists(f"{RUN}/weights/best.pt"), "NO best.pt - do not zip"
os.makedirs("out/weights",exist_ok=True); os.makedirs("out/results",exist_ok=True)
for f in ("best.pt","best.onnx"):
    s=f"{RUN}/weights/{f}"
    if os.path.exists(s): shutil.copy(s,f"out/weights/{f}")
for f in glob.glob(f"{RUN}/*.png")+glob.glob(f"{RUN}/*.csv")+glob.glob(f"{RUN}/*.yaml"):
    shutil.copy(f,"out/results/")
shutil.make_archive("model4_outputs","zip","out")
print("ZIP READY - weights:",os.listdir("out/weights"),"| results:",os.listdir("out/results"))